# The 2025 test half — scored once

Every number in this project so far comes from the **learning half**: 756 maps up to Masters
Toronto, cut into five folds. Several choices — the fading half-life, the combat-score repair,
logistic regression over trees — were made *after* seeing those fold scores. So the 61.1% is
mildly optimistic by construction: some of it is the model, and some of it is having been
chosen for doing well on those particular maps.

The **test half** is the 516 maps after Masters Toronto, which no model in this project has
been scored on. This notebook scores the frozen model on them, **once**.

**This notebook was committed to git before it was run.** Everything below — what is measured,
what it is compared against, how the margin of error is worked out, and what would count as
success — was fixed in that commit. The commit history is the proof that none of it was
chosen after seeing the answer.

## Decided in advance

1. **The model.** `make_final_model()`, fitted on all 756 learning maps. Nothing about it
   changes after this notebook runs, whatever the result.
2. **Its inputs on test maps.** Player form is built over the whole season in date order,
   backwards only: each test map sees every map played *before* it, including earlier test
   maps. That is how the model would be used for real — predicting next week's match, you
   know last week's results. The model's *weights* never see a test result. A check below
   confirms the learning maps' form is identical whether built from the learning half alone
   or from the whole season, so extending it cannot have leaked anything backwards.
3. **What is measured.** Accuracy and log loss on all 516 test maps. Nothing else.
4. **What it is compared against, on the same 516 maps.**
   - the **picker rule** — guess whoever chose the map, team A on deciders. The bar.
   - **always team A**
   - the **agent-picks model** (agents, map and who picked; the model without player stats),
     fitted on the same 756 learning maps.
5. **The margin of error.** 5,000 bootstrap resamples of whole *matches* — not maps, because
   maps in the same match are not independent: in about half of matches one
   team wins every map. Random seed 0. The 95% range is
   reported for the final model's accuracy, and for its lead over the picker rule.
6. **What counts as what, stated now:**
   - lead over the picker rule *and* the bottom of that lead's 95% range above zero →
     **beats the baseline**.
   - lead over the picker rule, but the range includes zero → **ahead, but not
     distinguishable from the baseline on this many maps**.
   - no lead → reported as that, plainly.
7. **Expected: lower than 61.1%.** Some of that figure is selection, as explained above. A
   drop of a few points is the honest cost of having chosen things; it is not a failure.
8. **No breakdowns.** Not by event, region, team or month. Every extra cut is another chance to
   find a lucky slice, and notebook 10 already showed there is nothing systematic to find.
   The whole test half, in one number per measure.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss

sys.path.insert(0, str(Path.cwd().parent / "src"))

pd.set_option("display.width", 200)

from dataset import build_map_table, split_season
from features import build_features, feature_columns
from form import player_form
from model import FINAL_COLUMNS, make_final_model, make_model

SEED, RESAMPLES = 0, 5000

## Build the inputs, and check nothing leaks backwards

In [2]:
maps = build_map_table()
learn, test = split_season(maps)
columns = feature_columns(maps)

form_season = player_form(maps, half_life=5)     # whole season, in date order, backwards only
form_learn = player_form(learn, half_life=5)     # learning half alone, as every notebook so far

# The guard: adding the test half must not change a single learning-half value.
pd.testing.assert_frame_equal(form_season.loc[form_learn.index, form_learn.columns], form_learn)
print("learning-half form is identical either way -- extending to the test half leaked nothing backwards")

x_learn, y_learn = build_features(learn, columns)
x_test, y_test = build_features(test, columns)
X_learn = pd.concat([x_learn, form_season.loc[learn.index]], axis=1)
X_test = pd.concat([x_test, form_season.loc[test.index]], axis=1)

print(f"learning maps {len(X_learn)}, test maps {len(X_test)} from {test['match_id'].nunique()} matches, "
      f"{test['played_at'].min():%d %b} to {test['played_at'].max():%d %b %Y}")
print(f"test maps with no rating form on a side: {X_test['rating_diff'].isna().sum()}, "
      f"of which combat score can stand in on {(X_test['rating_diff'].isna() & X_test['acs_diff'].notna()).sum()}")

learning-half form is identical either way -- extending to the test half leaked nothing backwards
learning maps 756, test maps 516 from 202 matches, 03 Jul to 05 Oct 2025
test maps with no rating form on a side: 0, of which combat score can stand in on 0


## Fit on the learning half, predict the test half

In [3]:
final = make_final_model().fit(X_learn[FINAL_COLUMNS].astype(float), y_learn)
agent_model = make_model(LogisticRegression(max_iter=5000)).fit(x_learn.astype(float), y_learn)

chance = {
    "final model": final.predict_proba(X_test[FINAL_COLUMNS].astype(float))[:, 1],
    "agent-picks model": agent_model.predict_proba(x_test.astype(float))[:, 1],
}
guess = {name: (p >= 0.5).astype(int) for name, p in chance.items()}
guess["picker rule"] = test["map_picked_by"].map({"team_a": 1, "team_b": 0}).fillna(1).astype(int).to_numpy()
guess["always team A"] = np.ones(len(test), dtype=int)

truth = y_test.to_numpy()
right = {name: (g == truth) for name, g in guess.items()}

## The result

In [4]:
results = pd.DataFrame({
    name: {"accuracy": right[name].mean(),
           "log loss": log_loss(truth, chance[name], labels=[0, 1]) if name in chance else np.nan}
    for name in right
}).T
print(f"{len(truth)} test maps.  Log loss for saying 50/50 to every map: 0.693\n")
print(results.to_string(formatters={"accuracy": "{:.1%}".format, "log loss": "{:.3f}".format}))

516 test maps.  Log loss for saying 50/50 to every map: 0.693

                  accuracy log loss
final model          54.5%    0.689
agent-picks model    50.8%    0.704
picker rule          52.9%      NaN
always team A        52.3%      NaN


## The margin of error

In [5]:
rng = np.random.default_rng(SEED)
match_of_map = test["match_id"].to_numpy()
matches = np.unique(match_of_map)
maps_in = {m: np.flatnonzero(match_of_map == m) for m in matches}

accuracy, lead = [], []
for _ in range(RESAMPLES):
    rows = np.concatenate([maps_in[m] for m in rng.choice(matches, size=len(matches), replace=True)])
    accuracy.append(right["final model"][rows].mean())
    lead.append(right["final model"][rows].mean() - right["picker rule"][rows].mean())

acc_lo, acc_hi = np.percentile(accuracy, [2.5, 97.5])
lead_point = right["final model"].mean() - right["picker rule"].mean()
lead_lo, lead_hi = np.percentile(lead, [2.5, 97.5])

print(f"final model accuracy        {right['final model'].mean():.1%}   95% range {acc_lo:.1%} to {acc_hi:.1%}")
print(f"lead over the picker rule   {lead_point * 100:+.1f} points   95% range {lead_lo * 100:+.1f} to {lead_hi * 100:+.1f}")

if lead_point > 0 and lead_lo > 0:
    verdict = "BEATS THE BASELINE"
elif lead_point > 0:
    verdict = "AHEAD, BUT NOT DISTINGUISHABLE FROM THE BASELINE ON THIS MANY MAPS"
else:
    verdict = "DOES NOT BEAT THE BASELINE"
print(f"\nby the rule decided in advance: {verdict}")

final model accuracy        54.5%   95% range 50.3% to 58.7%
lead over the picker rule   +1.6 points   95% range -3.2 to +6.4

by the rule decided in advance: AHEAD, BUT NOT DISTINGUISHABLE FROM THE BASELINE ON THIS MANY MAPS


## What this means

*Written after the cells above were run, in a separate commit. Everything above this line
was committed before running.*

**On 516 maps it had never been scored on, the frozen model got 54.5% right, against 52.9%
for the picker rule.** Its lead is 1.6 points, with a 95% range of −3.2 to +6.4. **By the rule
written before running, it is ahead, but not distinguishable from the baseline on this many
maps.**

**What held.** The agent-picks model came last again, at 50.8%, below "always team A". Its
log loss of 0.704 is worse than the 0.693 you get by saying 50/50 to every map, just as on
validation. The project's main negative finding — agent picks do not predict the winner —
is now confirmed on data nothing was tuned on.

**What didn't.** On validation the model led the picker rule by 5.6 points (61.1% against
55.5%). Here the lead is 1.6. The drop is bigger than the few points expected from having
chosen settings on validation scores: the test's 95% range for accuracy, 50.3% to 58.7%, does
not reach 61.1%.

**Part of the drop is the season, not the model.** The picker rule fell too, from 55.5% to
52.9%. Maps after Masters Toronto were closer to coin flips for everyone, not just for the
model.

**It is not a bug.** This was checked after the result, looking only at the test *inputs* —
no outcomes — so it could rule out broken plumbing without becoming another look at the
result:

| | Learning half | Test half |
|---|---|---|
| Rating gap, average (should be about 0) | +0.009 | −0.008 |
| Rating gap, spread | 0.101 | 0.092 |
| Rating gap, median size | 0.064 | 0.056 |
| Maps with no rating form on a side | 10.7% | 0% |
| Deciders | 17.1% | 18.6% |
| Model's typical distance from 50% | 5.8 points | 6.6 points |
| Share of maps called for team A | 49.2% | 46.7% |

The inputs look alike, and the model was *as sure of itself* on the test half as before — it
was simply right less often. That points to the link between recent form and winning being
weaker after Toronto, not to anything broken. (No test map lacked rating form on a side, so
the combat-score repair never came into play here.)

**Why the edge shrank is not known, and is not investigated here.** Choosing settings on 629
validation maps (winner's curse), a harder half-season with Champions in it, and the
squashing found in notebook 10 all fit. Testing any of them would mean cutting the test half
into slices after seeing its result — exactly what the plan ruled out. The 2026 season,
still untouched, is the place to find out.

**How this is presented** (the user's decision): test first, validation second. The model
scored 54.5% on unseen maps, ahead of a one-sentence rule but within the margin of error;
61.1% is the development figure, and the gap between them is the lesson.